In [3]:
!pip install transformers peft accelerate bitsandbytes
!pip install trl
!pip install -U bitsandbytes
!pip install jupyter ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 2.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 4.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 8.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 8.8 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 9.5 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53/53 [jupyter]0/53 [jupyterlab]server]


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments
import time

In [5]:
# --------------------------
# Device
# --------------------------
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", device)

Device: mps


In [7]:
# --------------------------
# Dataset
# --------------------------
dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "zh",
    split="train[0:]",
) # using a small subset for testing — adjust as needed for your experiments
print(f"Loaded {len(dataset)} samples.")


Loaded 20171 samples.


In [8]:
# --------------------------
# Model selection
# --------------------------
from transformers import BitsAndBytesConfig # Import BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config, # Pass the quantization config here
    device_map="auto", # Automatically map model layers to available devices
)

# model.to(device)    # No longer needed with device_map="auto"

Loading weights: 100%|██████████| 434/434 [00:10<00:00, 43.29it/s]


In [10]:
max_seq_length = 2048

def format_example(example):
    text = f"""### Question:
{example["Question"]}

### Response:
{example["Response"]}"""
    return {"text": text}

dataset = dataset.map(format_example)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_seq_length,
    )

dataset = dataset.map(tokenize_function, batched=True)
dataset = dataset.remove_columns(
    [col for col in dataset.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
#dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


Map:   0%|          | 0/20171 [00:00<?, ? examples/s]


KeyError: 'Question'

In [ ]:
# Prepare trainer

# --------------------------
# LoRA config
# --------------------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
) # these target modules are a good starting point for many models,
  # but you may want to adjust based on your specific architecture


model = get_peft_model(model, lora_config)  # wrap the model with LoRA adapter

# --------------------------
# Training args
# --------------------------
training_args = TrainingArguments(
    output_dir="./outputs.tmp",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=1e-4,
    logging_steps=100,
    lr_scheduler_type="cosine",             # Smooth decay
    weight_decay=0.01,                      # Light regularization
    bf16=True,                 # If using torch (Apple Silicon supports bf16 well)
    max_steps= 100,
    #evaluation_strategy="steps",            # Or "epoch" if you have val set
    save_strategy="steps",     # important
    save_steps=1000,           # save every 1000 steps
    save_total_limit=3,        # keep last 3 checkpoints
    report_to="none",
  )


trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)



In [ ]:
# --------------------------
# Train
# --------------------------
print("Start training")
start = time.perf_counter()
try:
    trainer.train()
except KeyboardInterrupt:
    print("Interrupted! Saving checkpoint...")
    trainer.save_model("./outputs/interrupted")

end = time.perf_counter()
print("End training.")

elapsed = end - start
print(f"Elapsed time: {elapsed:.3f} seconds")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Start training


Step,Training Loss
100,0.685685


End training.
Elapsed time: 686.545 seconds


In [ ]:
# --------------------------
# Save
# --------------------------
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
